In [17]:
import torch
import torch.nn as nn

In [18]:
block_size = 8 # n-1 gram
embedding_dim = 16
layer_size = 512
batch_size = 512
steps = 20000

In [19]:
# input & output initialization
with open("inputs-shakespeare.txt", "r") as f:
    data = f.read()
chars = sorted(set(data))
vocab_size = len(chars)

ctoi = {c: i for i, c in enumerate(chars)}
itoc = {i: c for i, c in enumerate(chars)}

train = data[:int(len(data) * 0.9)]
val = data[int(len(data) * 0.9):]

x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(0, block_size, 1)
y_train = torch.tensor([ctoi[ch] for ch in train[block_size:]], dtype=torch.long)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(0, block_size, 1)
y_val = torch.tensor([ctoi[ch] for ch in val[block_size:]], dtype=torch.long)

In [20]:
# model declaration
class MultiLayerPerceptron(nn.Module):
    def __init__(self, block_size, vocab_size, embedding_dim, layer_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Embedding(vocab_size, embedding_dim),
            nn.Flatten(),
            nn.Linear(in_features=embedding_dim * block_size, out_features=layer_size),
            nn.ReLU(),
            nn.Linear(in_features=layer_size, out_features=vocab_size)
        )

    def forward(self, x):
        return self.layers(x)

In [21]:
# model initialization
model = MultiLayerPerceptron(block_size, vocab_size, embedding_dim, layer_size)
# optimizer initialization
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

In [22]:
# training
for step in range(steps):
    optimizer.zero_grad()

    ix = torch.randint(0, len(x_train), (batch_size,))
    logits_train = model(x_train[ix])
    loss_train = nn.functional.cross_entropy(logits_train, y_train[ix])

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            logits_val = model(x_val)
            loss_val = nn.functional.cross_entropy(logits_val, y_val)
            print(f"Step {step}: train loss = {loss_train.item()}, val loss = {loss_val.item()}")

    loss_train.backward()
    optimizer.step()

Step 0: train loss = 4.218108654022217, val loss = 4.220717906951904
Step 1000: train loss = 1.6216230392456055, val loss = 1.9030635356903076
Step 2000: train loss = 1.604286551475525, val loss = 1.8352179527282715
Step 3000: train loss = 1.58924400806427, val loss = 1.798967719078064
Step 4000: train loss = 1.5521738529205322, val loss = 1.7752270698547363
Step 5000: train loss = 1.5887428522109985, val loss = 1.7598226070404053
Step 6000: train loss = 1.4678905010223389, val loss = 1.7473199367523193
Step 7000: train loss = 1.4921388626098633, val loss = 1.7483893632888794
Step 8000: train loss = 1.51933753490448, val loss = 1.7393550872802734
Step 9000: train loss = 1.4037829637527466, val loss = 1.7254029512405396
Step 10000: train loss = 1.5251250267028809, val loss = 1.7269196510314941
Step 11000: train loss = 1.425583004951477, val loss = 1.7289628982543945
Step 12000: train loss = 1.4191089868545532, val loss = 1.7242417335510254
Step 13000: train loss = 1.4361094236373901, va

In [23]:
# prediction
with torch.inference_mode():
    output = ['\n'] * block_size
    max_output_length = 1000
    while len(output) < max_output_length:
        i = torch.tensor([[ctoi[ch] for ch in output[-block_size:]]], dtype=torch.long)
        
        logits = model(i)
        P = torch.softmax(logits, dim=1)

        j = torch.multinomial(P, num_samples=1, replacement=True).item()

        output.append(itoc[j])

In [25]:
print(''.join(output[block_size:]))

RRDWEN:
You gnor at the kindrops mose crown,
To me: my lord, and the duke you no more.

SICINIUS:
God beoken feell not?
Come; let's hay say:
kepenme word go'tly life have Peedies, but not my bidit.
Pray you? if't they business very not are not exnor our it; and, my day;
Nay, and I sume being, and patience,
And all atching heregiant than my movest our pretty hif
Backinghourarring, like oar with a sichang melance a glore, call yourself prove thall neteless' down met.

CAMILLO:
Atay ree, so not this care; she free,
Wretch are him
in mine of ship; for this sid thou, but say he:
Our airads. most lady love power, Mowbray:
Pet it odded her fiten cares I have not for the be a tender bring not? Sir. Came gounted and stople in out
best
A prieft made shame.
Doverse than but to foight: neither; eat I noble look'd as: Margane'er did with stoop, now! Hark, by young Josuse I wan overer, go you think in our come
Flome to hear cheer;
If ireous.

See you take being false peers e'er sweety change
